# Test de la fonction d'association de station à des échangeurs

In [1]:
import geopandas as gpd
import pandas as pd

In [ ]:
def stations_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
    distance_max: float = 3000,
    col_station: str = "id_station",
    col_echangeur: str = "id_echangeur",
) -> pd.DataFrame:

    stations = stations[[col_station, "geometry"]].to_crs(epsg=2154)
    echangeurs = echangeurs[[col_echangeur, "geometry"]].to_crs(epsg=2154)

    result = gpd.sjoin_nearest(
        stations,
        echangeurs,
        how="inner",
        max_distance=distance_max,
        distance_col="distance_m",
    )

    result = result.loc[
        result["distance_m"] < distance_max,
        [col_station, col_echangeur, "distance_m"],
    ]

    return pd.DataFrame(result).reset_index(drop=True)

In [ ]:
def surfaces_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
    liens: pd.DataFrame,
    col_station: str = "id_station",
    col_echangeur: str = "id_echangeur",
    buffer_m: float = 100,
) -> gpd.GeoDataFrame:

    stations = stations.to_crs(epsg=2154)
    echangeurs = echangeurs.to_crs(epsg=2154)

    # Géométries des stations associées
    points_stations = stations[
        [col_station, "geometry"]
    ].merge(
        liens[[col_station, col_echangeur]],
        on=col_station,
        how="inner",
    )[[col_echangeur, "geometry"]]

    # Géométries des échangeurs
    points_echangeurs = echangeurs[
        [col_echangeur, "geometry"]
    ]

    # Regroupement des stations et des échangeurs
    points = gpd.GeoDataFrame(
        pd.concat(
            [points_stations, points_echangeurs],
            ignore_index=True,
        ),
        geometry="geometry",
        crs="EPSG:2154",
    )

    # Agrégation des points par échangeur
    surfaces = points.dissolve(by=col_echangeur)

    # Construction des enveloppes convexes
    surfaces["geometry"] = surfaces.geometry.convex_hull

    # Ajout d'une marge autour des enveloppes
    if buffer_m > 0:
        surfaces["geometry"] = surfaces.geometry.buffer(buffer_m)

    # Calcul des surfaces en km²
    surfaces["surface_km2"] = surfaces.geometry.area / 1e6

    return surfaces.reset_index()